<a href="https://colab.research.google.com/github/aronwilbert/COMP8420-Group-L-Healthcare/blob/main/COMP8420_Group_L_Healthcare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import pandas as pd
import kagglehub

# 1. Download the dataset folder
dataset_path = kagglehub.dataset_download('tboyle10/medicaltranscriptions')

# 2. Read the specific csv file from that folder
medical_transcriptions = pd.read_csv(os.path.join(dataset_path, 'mtsamples.csv'))

# 4. View the cleaned table structure
display(medical_transcriptions)

print(medical_transcriptions.shape)

100%|██████████| 4.85M/4.85M [00:00<00:00, 151MB/s]

Extracting files...


,Unnamed: 0,description,medical_specialty,sample_name,transcription,keywords
0,0,A 23-year-old white female presents with comp...,Allergy / Immunology,Allergic Rhinitis,"SUBJECTIVE:, This 23-year-old white female pr...","allergy / immunology, allergic rhinitis, aller..."
1,1,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 2,"PAST MEDICAL HISTORY:, He has difficulty climb...","bariatrics, laparoscopic gastric bypass, weigh..."
2,2,Consult for laparoscopic gastric bypass.,Bariatrics,Laparoscopic Gastric Bypass Consult - 1,"HISTORY OF PRESENT ILLNESS: , I have seen ABC ...","bariatrics, laparoscopic gastric bypass, heart..."
3,3,2-D M-Mode. Doppler.,Cardiovascular / Pulmonary,2-D Echocardiogram - 1,"2-D M-MODE: , ,1. Left atrial enlargement wit...","cardiovascular / pulmonary, 2-d m-mode, dopple..."
4,4,2-D Echocardiogram,Cardiovascular / Pulmonary,2-D Echocardiogram - 2,1. The left ventricular cavity size and wall ...,"cardiovascular / pulmonary, 2-d, doppler, echo..."
...,...,...,...,...,...,...
4994,4994,Patient having severe sinusitis about two to ...,Allergy / Immunology,Chronic Sinusitis,"HISTORY:, I had the pleasure of meeting and e...",NaN
4995,4995,This is a 14-month-old baby boy Caucasian who...,Allergy / Immunology,Kawasaki Disease - Discharge Summary,"ADMITTING DIAGNOSIS: , Kawasaki disease.,DISCH...","allergy / immunology, mucous membranes, conjun..."
4996,4996,A female for a complete physical and follow u...,Allergy / Immunology,Followup on Asthma,"SUBJECTIVE: , This is a 42-year-old white fema...",NaN
4997,4997,Mother states he has been wheezing and coughing.,Allergy / Immunology,Asthma in a 5-year-old,"CHIEF COMPLAINT: , This 5-year-old male presen...",NaN


(4999, 6)

In [3]:
# 1. Check for standard NaN / Null values in all columns
print("--- NaN / Null Value Counts ---")
print(medical_transcriptions.isna().sum())

# 2. Check for empty strings or whitespace-only strings in text columns
print("\n--- Empty String Counts ('') ---")
for col in medical_transcriptions.columns:
  empty_count = medical_transcriptions[col].apply(lambda x: str(x).strip() == "").sum()
  print(f"{col}: {empty_count}")

--- NaN / Null Value Counts ---
Unnamed: 0              0
description             0
medical_specialty       0
sample_name             0
transcription          33
keywords             1068
dtype: int64

--- Empty String Counts ('') ---
Unnamed: 0: 0
description: 6
medical_specialty: 0
sample_name: 0
transcription: 0
keywords: 81


In [4]:
# Isolate the 33 rows where transcription is missing
missing_transcriptions = medical_transcriptions[medical_transcriptions['transcription'].isna()]

# Display the entire subset to inspect all columns
display(missing_transcriptions)

,Unnamed: 0,description,medical_specialty,sample_name,transcription,keywords
97,97,Inguinal orchiopexy procedure.,Urology,Inguinal orchiopexy,NaN,"urology, inguinal orchiopexy, keith needles, a..."
116,116,Inguinal hernia hydrocele repair.,Urology,Hydrocele Repair,NaN,"urology, inguinal hernia, external oblique, he..."
205,205,Vaginal Hysterectomy. A weighted speculum wa...,Surgery,Vaginal Hysterectomy,NaN,"surgery, omentum, massachusetts, vaginal hyste..."
263,263,Total Abdominal Hysterectomy (TAH). An incis...,Surgery,Total Abdominal Hysterectomy,NaN,"surgery, fundus, double-toothed tenaculum, mus..."
459,459,Parotidectomy procedure,Surgery,Parotidectomy,NaN,"surgery, parotidectomy, mixter clamp, auditory..."
622,622,Laparoscopy. The cervix was grasped with a s...,Surgery,Laparoscopy - 1,NaN,"surgery, uterus, cervix, vaginal, single tooth..."
628,628,Laparoscopy. An incision was made in the umb...,Surgery,Laparoscopy - 2,NaN,"surgery, umbilicus, trocar, falope, laparoscop..."
680,680,Inguinal orchiopexy procedure.,Surgery,Inguinal orchiopexy,NaN,"surgery, inguinal orchiopexy, keith needles, a..."
729,729,Inguinal hernia hydrocele repair.,Surgery,Hydrocele Repair,NaN,"surgery, inguinal hernia, external oblique, he..."
871,871,Common description of EGD,Surgery,EGD Template - 4,NaN,"surgery, lateral supine position, stomach, duo..."


In [7]:
# 1. Drop the 33 missing transcription rows
df_clean = medical_transcriptions.dropna(subset=['transcription']).copy()

# 2. Check the remaining NaN count for each column
print("--- Remaining NaN Counts Per Column ---")
print(df_clean.isna().sum())

--- Remaining NaN Counts Per Column ---
Unnamed: 0              0
description             0
medical_specialty       0
sample_name             0
transcription           0
keywords             1068
dtype: int64


To give you the short, straightforward answer: **You only need to focus on two columns for the core machine learning tasks: `transcription` and `medical_specialty`.** You can completely ignore the other four columns for your pipeline models. Here is exactly why, column by column, so you know what to do with them:

---

### 1. The Core Target: `medical_specialty`

* **Status:** **CRITICAL**
* **Why:** This is your **label** (the ground truth target). Abrar’s text classifier needs this column to learn what department a file belongs to. If the model reads keywords like *"echocardiogram"* in the transcription, it checks this column to confirm: *"Ah-ha, that means it's Cardiovascular/Pulmonary."*

### 2. The Core Input: `transcription`

* **Status:** **CRITICAL**
* **Why:** This is your **feature** (the input text). This column contains the rich, paragraph-length patient histories, physical exams, and clinical narratives. Your privacy filter will clean *this* text, and Abrar's medical NER will extract diseases and medicines from *this* text.

---

### What to do with the other 4 columns? (Ignore them)

* **`Unnamed: 0`:** This is just a row index number (0, 1, 2, 3...). It has no clinical meaning and will confuse an AI classifier.
* **`description`:** As we discovered during your data audit, these are just short summaries or duplicate template definitions. Since the `transcription` column already contains all of this information in much better detail, keeping the description column is entirely redundant.
* **`sample_name`:** This is just the title of the medical template used (e.g., *"Allergic Rhinitis"*). Using it would be cheating for the classifier because it gives away the answer instantly, but it doesn't provide the messy text the assignment requires you to analyze.
* **`keywords`:** This column has over 1,068 missing (`NaN`) rows. It’s highly incomplete, and your medical NER model is already automatically generating much better keywords anyway.

---

### The Cleanest Next Step For Your Notebook

To keep your notebook clear and fast, use this code to drop the 33 missing rows and strip away the useless columns all at once. This leaves you with a clean, focused dataset:

```python
# 1. Drop the 33 rows where transcription is missing
df_clean = medical_transcriptions.dropna(subset=['transcription']).copy()

# 2. Keep only the two columns needed for your machine learning pipeline
df_pipeline = df_clean[['medical_specialty', 'transcription']]

# 3. View your streamlined data shape
print(f"Pipeline ready! Data shape: {df_pipeline.shape}")
display(df_pipeline.head())

```

This ensures your workspace is completely optimized. Would you like to check the breakdown of how many patient files exist for each department in `medical_specialty` next?

In [8]:
# 1. Drop the 33 rows where transcription is missing
df_clean = medical_transcriptions.dropna(subset=['transcription']).copy()

# 2. Keep only the two columns needed for your machine learning pipeline
df_pipeline = df_clean[['medical_specialty', 'transcription']]

# 3. View your streamlined data shape
print(f"Pipeline ready! Data shape: {df_pipeline.shape}")
display(df_pipeline.head())

Pipeline ready! Data shape: (4966, 2)


medical_specialty  \
0         Allergy / Immunology   
1                   Bariatrics   
2                   Bariatrics   
3   Cardiovascular / Pulmonary   
4   Cardiovascular / Pulmonary   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                